# Gao et al. (2018): the CN recordings, from Zenodo

**Paper.** Gao Z, Davis C, Thomas AM, Economo MN, Abrego AM, Svoboda K, De Zeeuw CI, Li N (2018). A
cortico-cerebellar loop for motor planning. *Nature* 563, 113-116.
[doi:10.1038/s41586-018-0633-x](https://doi.org/10.1038/s41586-018-0633-x). Full text of the author
manuscript (PMC6212318) and the publisher's Supplementary Information.

**Data.** Dandiset 000007 (version 0.220126.1903, see `reproduce_gao_2018.ipynb`) holds only the ALM
recordings. The CN recordings of Figures 2 and 3 and Extended Data Figures 3 to 6 are deposited on
Zenodo as NWB files in one zip archive, [doi:10.5281/zenodo.6647629](https://doi.org/10.5281/zenodo.6647629)
(`datafiles_NWB.zip`, 11.9 GB), found through CRCNS alm-6, which the paper's reporting summary names.
The archive is streamed with HTTP range requests and inflated in memory (`gao2018_cn.py`); the ALM
fraction of fig2-cn-preparatory-fraction is computed from the dandiset. The claims here are counted
apart from those on the dandiset.

**Method.** Trial selection, epochs and the selectivity and photo-inhibition analyses follow the
paper's CN scripts (`analyses_scripts.zip` in the MATLAB record doi:10.5281/zenodo.6653158).

| Claim | Status |
|---|---|
| fig2-cn-unit-counts | tested |
| fig2-cn-selective-fraction | tested |
| fig2-cn-preparatory-fraction | tested |
| fig2-cn-perimovement-fraction | tested |
| ed4e-cn-predict-movement | tested |
| ed4g-fn-contra-late-delay | tested |
| fig3b-cn-modulated-fraction | tested |
| fig3b-cn-net-increase | tested |
| fig3-cn-latency | tested |
| fig3c-cn-selectivity-abolished | tested |
| ed6-cn-either-hemisphere | tested |
| ed4c-lick-modulation | blocked, too large |

Streaming the archive takes most of the run, about 12 to 15 minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import gao2018
import gao2018_cn as cn

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
entries = cn.archive_entries()
print(f"Zenodo record {cn.ZENODO_DOI}: {len(entries)} NWB files, {sum(e.compress_size for e in entries) / 1e9:.2f} GB "
      f"compressed, {sum(e.file_size for e in entries) / 1e9:.2f} GB inflated")
cn_sessions, timing = cn.load_all(entries)
print(f"streamed and parsed {timing['files']} files in {timing['seconds'] / 60:.1f} min "
      f"({timing['compressed_bytes'] / timing['seconds'] / 1e6:.1f} MB/s over {cn.CONNECTIONS} connections)")

## Data check

Each check of the record's description against the values is stated under its output.

In [ ]:
units = pd.DataFrame([{"file": s["file"], "mouse": s["mouse"], "unit": i, "location": loc, "spikes": len(sp)}
                      for s in cn_sessions for i, (loc, sp) in enumerate(zip(s["location"], s["spikes"]))])
sess = pd.DataFrame([{"file": s["file"], "mouse": s["mouse"], "trials": len(s["trials"]), "good": int(s["trials"].good.sum()),
                      "units": len(s["spikes"]), "codes": " ".join(f"{c:g}:{n}" for c, n in
                                                                   s["trials"].stim.value_counts(dropna=False).sort_index().items())}
                     for s in cn_sessions])
print("Mice:", sess.mouse.nunique(), "; sessions:", len(sess), "; units:", len(units))
print("\nUnits by location:", units.location.value_counts(dropna=False).to_dict())
print("\nBy mouse:")
print(units.groupby(["mouse", "location"]).size().unstack(fill_value=0).assign(sessions=sess.groupby("mouse").size()).to_string())
alltr = pd.concat([s["trials"].assign(mouse=s["mouse"]) for s in cn_sessions], ignore_index=True)
print("\nTrials:", len(alltr), "; performing (GoodTrials = 1):", int(alltr.good.sum()))
print("PhotostimulationType values:", alltr.stim.value_counts(dropna=False).sort_index().to_dict())
print("Mice with ALM photo-inhibition trials (codes 7 to 10):", sorted(alltr[alltr.stim.isin([7, 8, 9, 10])].mouse.unique()))
print("Spike times sorted after ordering:", all(np.all(np.diff(sp) >= 0) for s in cn_sessions for sp in s["spikes"]))
g = alltr[alltr.good == 1]
print("\nDelay epoch (cue - pole out) of performing trials without early licks, median per mouse (s):",
      alltr[(alltr.good == 1) & (alltr.LickEarly == 0)].assign(d=lambda t: t.cue - t.pole_out).groupby("mouse").d.median().round(2).to_dict())
print("Performing trials, epoch lengths (s): sample", (g.pole_out - g.pole_in)[g.LickEarly == 0].round(3).agg(["min", "median", "max"]).tolist(),
      "; delay", (g.cue - g.pole_out)[g.LickEarly == 0].round(3).agg(["min", "median", "max"]).tolist(),
      "; pole in after trial start", g.pole_in.round(3).agg(["min", "median", "max"]).tolist())

The record holds 185 sessions from 18 mice, as its description says; the Methods give 169
sessions. In the sessions of the 2016 mice (ANM ids) the delay epoch is 1.7 s, not the 1.3 s of
the Methods; the epochs are taken from each trial's own times. The unit counts by location are set against the paper in fig2-cn-unit-counts. Pole in,
pole out and cue are relative to each trial's start, as the column descriptions say, and the
sample and delay epochs are 1.3 s in performing trials without early licks. ALM photo-inhibition
codes occur only in the 8 VGAT-ChR2 mice (BAYLORNL3 to NL11 in Supplemental Table 1).

In [ ]:
# Duplicated recordings: identical spike trains across all units, and the two penetrations of one day.
import hashlib
from collections import defaultdict
digests = defaultdict(list)
for s in cn_sessions:
    for i, sp in enumerate(s["spikes"]):
        if len(sp):
            digests[hashlib.md5(np.round(sp, 5).tobytes()).hexdigest()].append((s["file"], i))
dups = [v for v in digests.values() if len(v) > 1]
print("Groups of identical spike trains:", len(dups), dups[:10])
same_day = sess[sess.file.str.contains("pene")]
print("Files with several penetrations on one day:", same_day.file.tolist())
starts = defaultdict(list)
for s in cn_sessions:
    starts[(s["mouse"], round(float(s["trials"].start.iloc[0]), 2), len(s["trials"]))].append(s["file"])
print("Files sharing a mouse, first trial start and trial count:", [v for v in starts.values() if len(v) > 1])

In [ ]:
# Clock check: CN firing around pole in, lick right against lick left (correct control trials), in
# 200 ms windows; the population difference should be near 0 before pole in and grow after it.
offs = np.round(np.arange(-0.5, 3.9, 0.2), 2)
diffs = []
for s in cn_sessions:
    t = s["trials"]
    ok = cn.usable(s) & (t.stim == 0) & (t.outcome == "hit")
    for i, loc in enumerate(s["location"]):
        if loc in (None, "outside"):
            continue
        st = cn.stable(s, i)
        r, l = (ok & st & (t.trial_type == "R")).values, (ok & st & (t.trial_type == "L")).values
        if min(r.sum(), l.sum()) < 10:
            continue
        x = cn.window_counts(s, i, "pole_in", offs + 0.1, 0.2) / 0.2
        tt = stats.ttest_ind(x[r], x[l], axis=0)
        diffs.append(np.nan_to_num(tt.pvalue, nan=1.0) < 0.05)
diffs = np.array(diffs)
print(f"CN units tested: {len(diffs)}; fraction with lick right and lick left differing (p < 0.05) per window, by window start:")
print(pd.Series(diffs.mean(0), index=offs).round(3).to_string())

Before pole in the fraction of CN units distinguishing the trial types is near the 5% expected by
chance, and it rises through the sample and delay epochs, so the trial times and the spikes are
on one clock at the scale of the epochs. ALM photo-inhibition onset is checked at a finer scale in
fig3-cn-latency.

## Selectivity of CN neurons

The test of `analysis_compile_all_DCN_data.m`: correct control trials without early licks, in
performing trials and the unit's stable trials, a two-tailed t-test of lick right against lick
left rates in the sample, delay and response epochs.

In [ ]:
rows = []
for s in cn_sessions:
    t = s["trials"]
    base = cn.usable(s) & (t.stim == 0)
    for i, loc in enumerate(s["location"]):
        st = cn.stable(s, i)
        r_ = cn.epoch_rates(s, i)
        R = (base & st & (t.outcome == "hit") & (t.trial_type == "R")).values
        L = (base & st & (t.outcome == "hit") & (t.trial_type == "L")).values
        row = {"file": s["file"], "mouse": s["mouse"], "unit": i, "location": loc, "n_R": int(R.sum()), "n_L": int(L.sum())}
        for e in ("sample", "delay", "response"):
            row[f"p_{e}"] = stats.ttest_ind(r_[e].values[R], r_[e].values[L]).pvalue if min(R.sum(), L.sum()) >= 2 else np.nan
        rows.append(row)
cn_sel = pd.DataFrame(rows)
for e in ("sample", "delay", "response"):
    cn_sel[f"sig_{e}"] = cn_sel[f"p_{e}"] < 0.05
cn_sel["selective"] = cn_sel[["sig_sample", "sig_delay", "sig_response"]].any(axis=1)
cn_sel["preparatory"] = cn_sel[["sig_sample", "sig_delay"]].any(axis=1)
in_cn = cn_sel.location.isin(["FN", "IP", "DN"])
print("CN units:", int(in_cn.sum()), "; untested in some epoch:", int(cn_sel[in_cn][["p_sample", "p_delay", "p_response"]].isna().any(axis=1).sum()))
print(cn_sel[in_cn].groupby("location")[["selective", "preparatory", "sig_response"]].mean().round(3)
      .assign(units=cn_sel[in_cn].groupby("location").size()).to_string())

## fig2-cn-unit-counts

> 878 CN-region single units were recorded in 18 mice across 169 sessions, 314 from outside the
> CN, 268 in the fastigial, 139 in the interposed and 157 in the dentate nucleus.

In [ ]:
cnt = pd.DataFrame({"expected": [878, 314, 268, 139, 157, 169],
                    "measured": [len(units), int((units.location == "outside").sum()), int((units.location == "FN").sum()),
                                 int((units.location == "IP").sum()), int((units.location == "DN").sum()), len(sess)]},
                   index=["single units", "outside the CN", "fastigial", "interposed", "dentate", "sessions"])
cnt["tolerance"] = (cnt.expected * 0.1).round(1)
cnt["within"] = (cnt.measured - cnt.expected).abs() <= cnt.tolerance
print(cnt.to_string())
print("Within tolerance:", int(cnt.within.sum()), "of", len(cnt), "; units in the CN:", int(in_cn.sum()), "(Results: 564)")
m30 = in_cn & (cn_sel.n_R >= 30) & (cn_sel.n_L >= 30)
print("Variant, CN units with at least 30 correct control trials of each type:", int(m30.sum()))
print("Sessions with no units:", int((sess.units == 0).sum()), "; with only units outside the CN:",
      int(sum(all(l == "outside" for l in s["location"]) for s in cn_sessions if s["location"])))

**Result: not reproduced.** Only the session count is within tolerance (185 against 169). The
record holds 1212 units against 878: 455 outside the CN (314), 377 fastigial (268), 178
interposed (139) and 202 dentate (157), 757 in the CN against the Results' 564. 457 CN units
have at least 30 correct control trials of each type. The paper's unit selection is not
described.

## fig2-cn-selective-fraction

> A similar proportion of CN neurons as of ALM neurons showed trial-type selectivity (416/564).

In [ ]:
v = cn_sel.selective[in_cn].mean()
print(f"CN units selective: {int(cn_sel.selective[in_cn].sum())}/{int(in_cn.sum())} = {v:.3f} (expected 0.738 +/- 0.074; within: {abs(v - 0.738) <= 0.074})")
print("By mouse:", cn_sel[in_cn].groupby("mouse").selective.mean().round(2).to_dict())
m30 = in_cn & (cn_sel.n_R >= 30) & (cn_sel.n_L >= 30)
print(f"Variant, CN units with at least 30 correct control trials of each type (the scripts' rule for their population "
      f"analyses): {int(m30.sum())} units, selective {cn_sel.selective[m30].mean():.3f}, preparatory {cn_sel.preparatory[m30].mean():.3f}, "
      f"peri-movement {cn_sel.sig_response[m30].mean():.3f}")
print("Correct control trials of each type per CN unit, quartiles (lick right):", cn_sel.n_R[in_cn].quantile([0.25, 0.5, 0.75]).tolist())
rng = np.random.default_rng(601)
null = []
for s in cn_sessions:
    t = s["trials"].copy()
    t["trial_type"] = rng.permutation(t.trial_type.values)
    sh = dict(s, trials=t)
    base = cn.usable(sh) & (t.stim == 0)
    for i, loc in enumerate(s["location"]):
        if loc not in ("FN", "IP", "DN"):
            continue
        st = cn.stable(sh, i)
        r_ = cn.epoch_rates(sh, i)
        R = (base & st & (t.outcome == "hit") & (t.trial_type == "R")).values
        L = (base & st & (t.outcome == "hit") & (t.trial_type == "L")).values
        if min(R.sum(), L.sum()) < 2:
            continue
        null.append(any(stats.ttest_ind(r_[e].values[R], r_[e].values[L]).pvalue < 0.05 for e in ("sample", "delay", "response")))
print(f"Shuffled trial types: {np.mean(null):.3f} of CN units 'selective' (the chance level of this test)")

**Result: not reproduced.** 448 of 757 CN units (0.592) are selective, outside 0.738 +/- 0.074;
the chance level of the test is 0.127. Restricted to the 457 units with at least 30 correct
control trials of each type, the scripts' rule for their population analyses, the fraction is
0.757, within tolerance, which suggests the paper's 564 neurons were selected by trial count or
a similar rule that the Methods do not state. The fraction ranges from 0.07 to 0.93 across mice.

## fig2-cn-preparatory-fraction

> Preparatory activity was more prevalent in ALM than in CN (59% against 35% of neurons, p <
> 0.001, chi-square test).

The ALM fraction is computed here from the dandiset, as reading (a) of the main notebook (the
pyramidal neurons of the 9 photo-activation mice).

In [ ]:
alm_sessions = gao2018.load_all()
alm = pd.concat([gao2018.selectivity_tests(s, unit_mask=s["cell_type"] == "pyramidal") for s in alm_sessions
                 if s["group"] in ("FN", "DN")], ignore_index=True)
alm_prep = int(((alm.p_sample < 0.05) | (alm.p_delay < 0.05)).sum())
cn_prep = int(cn_sel.preparatory[in_cn].sum())
v = cn_prep / in_cn.sum()
table = np.array([[alm_prep, len(alm) - alm_prep], [cn_prep, int(in_cn.sum()) - cn_prep]])
chi2, p, _, _ = stats.chi2_contingency(table)
print(f"ALM (dandiset, {gao2018.DANDISET_ID} version {gao2018.DANDISET_VERSION}): {alm_prep}/{len(alm)} = {alm_prep / len(alm):.3f}")
print(f"CN: {cn_prep}/{int(in_cn.sum())} = {v:.3f} (expected 0.35 +/- 0.035; within: {abs(v - 0.35) <= 0.035})")
print(f"chi-square = {chi2:.1f}, p = {p:.2g}; ALM higher: {alm_prep / len(alm) > v}")
print("CN by nucleus:", cn_sel[in_cn].groupby("location").preparatory.mean().round(3).to_dict())
m30 = in_cn & (cn_sel.n_R >= 30) & (cn_sel.n_L >= 30)
print(f"Variant, CN units with at least 30 correct control trials of each type: {cn_sel.preparatory[m30].mean():.3f}")

**Result: partially reproduced.** Preparatory activity is more prevalent in ALM (621 of 1112,
0.558) than in CN (204 of 757, 0.269), chi-square p = 8.7e-35, but the CN fraction is outside
0.35 +/- 0.035. With at least 30 correct trials of each type the CN fraction is 0.372, within
tolerance.

## fig2-cn-perimovement-fraction

> Peri-movement activity was equally prevalent in ALM and CN (67% against 66%).

In [ ]:
v = cn_sel.sig_response[in_cn].mean()
alm_resp = float((alm.p_response < 0.05).mean())
print(f"CN response-epoch selective: {v:.3f} (expected 0.66 +/- 0.066; within: {abs(v - 0.66) <= 0.066}); ALM (dandiset) {alm_resp:.3f}")
t2 = np.array([[int((alm.p_response < 0.05).sum()), int((alm.p_response >= 0.05).sum() + alm.p_response.isna().sum())],
               [int(cn_sel.sig_response[in_cn].sum()), int((~cn_sel.sig_response[in_cn]).sum())]])
print(f"chi-square ALM against CN: p = {stats.chi2_contingency(t2)[1]:.2g}")
m30 = in_cn & (cn_sel.n_R >= 30) & (cn_sel.n_L >= 30)
print(f"Variant, CN units with at least 30 correct control trials of each type: {cn_sel.sig_response[m30].mean():.3f}")

**Result: not reproduced.** 0.511 of CN units are selective in the response epoch, outside 0.66
+/- 0.066, and lower than ALM's 0.629 (chi-square p = 5.7e-07), where the paper finds them equal.
With at least 30 correct trials of each type the CN fraction is 0.670, within tolerance.

## ed4e-cn-predict-movement

> Most CN neurons with delay selectivity predicted the upcoming licking direction, switching
> their trial-type preference on error trials (67%, n = 73 neurons tested on more than 3 error
> trials).

In [ ]:
rows = []
for s in cn_sessions:
    t = s["trials"]
    base = cn.usable(s) & (t.stim == 0)
    sel_s = cn_sel[cn_sel.file == s["file"]].set_index("unit")
    for i, loc in enumerate(s["location"]):
        if loc not in ("FN", "IP", "DN") or not sel_s.loc[i, "sig_delay"]:
            continue
        st = cn.stable(s, i)
        d = cn.epoch_rates(s, i)["delay"].values
        m = lambda o, side: (base & st & (t.outcome == o) & (t.trial_type == side)).values
        if min(m("error", "R").sum(), m("error", "L").sum()) <= 3:
            continue
        c_sel = d[m("hit", "R")].mean() - d[m("hit", "L")].mean()
        e_sel = d[m("error", "R")].mean() - d[m("error", "L")].mean()
        rows.append({"mouse": s["mouse"], "location": loc, "ratio": e_sel / c_sel})
ed4e = pd.DataFrame(rows)
v = float((ed4e.ratio < 0).mean())
print(f"Delay-selective CN neurons with more than 3 error trials of each type: {len(ed4e)} (expected 73 +/- 7.3)")
print(f"Fraction with a negative error/correct ratio (predicting movement): {v:.3f} (expected 0.67 +/- 0.067; within: {abs(v - 0.67) <= 0.067})")
print("Median ratio:", round(float(ed4e.ratio.median()), 2), "; by nucleus:", ed4e.groupby("location").ratio.apply(lambda r: round(float((r < 0).mean()), 2)).to_dict())

**Result: not reproduced.** Of 117 delay-selective CN neurons with more than 3 error trials of
each type (against 73), 0.496 have a negative error to correct ratio, outside 0.67 +/- 0.067; the
median ratio is 0.02.

## ed4g-fn-contra-late-delay

> In the fastigial nucleus, equal numbers of neurons preferred contralateral or ipsilateral
> movement in the early delay, and a population preference for contralateral movement developed
> right before movement initiation.

In [ ]:
rng = np.random.default_rng(602)
rows = []
for s in cn_sessions:
    t = s["trials"]
    base = cn.usable(s) & (t.stim == 0) & (t.outcome == "hit")
    for i, loc in enumerate(s["location"]):
        if loc != "FN":
            continue
        st = cn.stable(s, i)
        R, L = (base & st & (t.trial_type == "R")).values, (base & st & (t.trial_type == "L")).values
        if min(R.sum(), L.sum()) < 2:
            continue
        early = cn.window_counts(s, i, "pole_out", [0.25], 0.5)[:, 0]
        late = cn.window_counts(s, i, "cue", [-0.25], 0.5)[:, 0]
        for w, x in (("early", early), ("late", late)):
            p = stats.ttest_ind(x[R], x[L]).pvalue
            if p < 0.05:
                rows.append({"mouse": s["mouse"], "window": w, "contra": x[R].mean() > x[L].mean()})
fn_pref = pd.DataFrame(rows)
per_mouse = fn_pref.groupby(["window", "mouse"]).contra.mean().unstack("window")
print("Significant FN neurons per window:", fn_pref.groupby("window").size().to_dict(), "; mice:", per_mouse.index.nunique(), "(paper, 15)")
print("Fraction contralateral (lick right) per mouse:"); print(per_mouse.round(2).to_string())
res = {}
for w in ("early", "late"):
    x = per_mouse[w].dropna().values
    boot = np.array([x[rng.integers(0, len(x), len(x))].mean() for _ in range(10000)])
    res[w] = (x.mean(), float((boot <= 0.5).mean()), float(2 * min((boot <= 0.5).mean(), (boot >= 0.5).mean())))
    print(f"{w}: mean over {len(x)} mice {x.mean():.3f}; one-tailed p (not above 0.5) {res[w][1]:.4f}; two-tailed p {min(res[w][2], 1):.4f}; "
          f"pooled over neurons {fn_pref[fn_pref.window == w].contra.mean():.3f}")

**Result: reproduced.** In the last 500 ms of the delay a mean of 0.650 of significant fastigial
neurons per mouse prefer contralateral licking (13 mice, one-tailed p = 0.033), and in the first
500 ms 0.639 do (11 mice), not different from 0.5 (two-tailed p = 0.219). The paper's p = 0.0001
for the late window is not matched; the early fraction is above one half without significance.

## CN during ALM photo-inhibition (Figure 3a to 3c)

In [ ]:
def cn_units_with(code_set):
    out = []
    for s in cn_sessions:
        t = s["trials"]
        if not t.stim.isin(code_set).any():
            continue
        base = (cn.usable(s) & t.outcome.isin(["hit", "error"])).values
        for i, loc in enumerate(s["location"]):
            if loc in ("FN", "IP", "DN"):
                out.append({"s": s, "i": i, "mouse": s["mouse"], "location": loc, "use": base & cn.stable(s, i)})
    return out


vgat_units = cn_units_with({9})
mod = []
for u in vgat_units:
    s, i, use = u["s"], u["i"], u["use"]
    t = s["trials"]
    d = cn.epoch_rates(s, i)["delay"].values * (t.cue - t.pole_out).values
    c, st = d[use & (t.stim.values == 0)], d[use & (t.stim.values == 9)]
    mod.append({"p": stats.ttest_ind(c, st).pvalue if min(len(c), len(st)) >= 2 else np.nan,
                "diff": (st.mean() - c.mean()) / 1.3 if len(st) else np.nan, "n9": len(st)})
mod = pd.DataFrame(mod)
print("CN units in sessions with bilateral ALM photo-inhibition (code 9):", len(vgat_units), "(paper, 389)")
print("Tested (2 or more trials in each condition):", int(mod.p.notna().sum()))

## fig3b-cn-modulated-fraction

> Activity of many CN neurons was rapidly altered by bilateral ALM photo-inhibition (185/389, p
> < 0.01, two-tailed t-test).

In [ ]:
sig = mod.p < 0.01
v = sig.sum() / mod.p.notna().sum()
print(f"Modulated: {int(sig.sum())}/{int(mod.p.notna().sum())} = {v:.3f} (expected 0.476 +/- 0.048; within: {abs(v - 0.476) <= 0.048})")
print("By nucleus:", pd.Series(sig.values, index=[u["location"] for u in vgat_units]).groupby(level=0).mean().round(3).to_dict())
print("By mouse:", pd.Series(sig.values, index=[u["mouse"] for u in vgat_units]).groupby(level=0).mean().round(3).to_dict())
print("Variant, p < 0.05:", round(float((mod.p < 0.05).sum() / mod.p.notna().sum()), 3))

**Result: not reproduced.** 145 of 517 CN units (0.280) in sessions with bilateral ALM photo-
inhibition are modulated at p < 0.01, outside 0.476 +/- 0.048; at p < 0.05 the fraction is 0.412.
The record has 563 CN units in these sessions, against the paper's 389. The fraction is 0.435 in
the dentate, 0.178 in the fastigial and 0.186 in the interposed nucleus.

## fig3b-cn-net-increase

> ALM photo-inhibition caused more CN neurons to increase than to decrease their activity
> (123/185 increase, p < 0.001, binomial test).

In [ ]:
up = int((mod["diff"][sig] > 0).sum())
v = up / sig.sum()
print(f"Increased {up}, decreased {int(sig.sum()) - up}; fraction increased {v:.3f} (expected 0.665 +/- 0.066; within: "
      f"{abs(v - 0.665) <= 0.066}); binomial p = {stats.binomtest(up, int(sig.sum())).pvalue:.2g}")

**Result: reproduced.** Of the 145 modulated CN units, 100 increase their delay-epoch rate
(0.690, within 0.665 +/- 0.066; binomial p = 5.7e-06).

## fig3-cn-latency

> The CN response to ALM photo-inhibition had a latency of 10.7 +/- 4.0 ms (mean +/- s.e.m.).

In [ ]:
rng = np.random.default_rng(603)
lat_centres = np.round(np.arange(-0.015, 0.1 + 1e-9, 0.001), 4)
curves = []
for u, k, dsign in zip(vgat_units, sig.values, np.sign(mod["diff"].values)):
    if not k:
        continue
    s, i, use = u["s"], u["i"], u["use"]
    t = s["trials"]
    x = cn.window_counts(s, i, "pole_out", lat_centres, 0.01) / 0.01
    curves.append(dsign * (x[use & (t.stim.values == 9)].mean(0) - x[use & (t.stim.values == 0)].mean(0)))
curves = np.array(curves)
mean, sd, onsets = gao2018.onset_latency(curves, lat_centres, rng)
mean10, sd10, _ = gao2018.onset_latency(curves, lat_centres, rng, n_boot=2000, run=10)
print(f"{len(curves)} modulated CN neurons; onset {mean * 1000:.1f} +/- {sd * 1000:.1f} ms (expected 10.7 +/- 4.0; within: "
      f"{abs(mean * 1000 - 10.7) <= 4.0}); resampled onsets before 0 ms: {float((onsets < 0).mean()):.3f}")
print(f"Variant, sustained onset (10 consecutive bins): {mean10 * 1000:.1f} +/- {sd10 * 1000:.1f} ms")
print("Population change (spikes/s) at bin centres -10, 0, 5, 10, 15, 20, 30 ms:",
      [round(float(curves.mean(0)[np.argmin(np.abs(lat_centres - x / 1000))]), 2) for x in (-10, 0, 5, 10, 15, 20, 30)])
prec = float(np.mean([np.mean(np.abs(s["trials"].pole_out.dropna().values * 100 - np.round(s["trials"].pole_out.dropna().values * 100)) < 1e-6)
                      for s in cn_sessions if s["trials"].stim.isin([9]).any()]))
print(f"Fraction of pole-out times on a 10 ms grid in these sessions: {prec:.2f}")

**Result: not reproduced.** The first-crossing onset is 0.2 +/- 13.7 ms, outside 10.7 +/- 4.0
ms, and 47% of resampled onsets fall before the photostimulus; the population change is 1.14
spikes/s at -10 ms and 0.29 at 5 ms, and rises to 3.16 at 15 ms. The sustained onset is 14.3 +/-
2.0 ms. In these sessions 96% of pole-out times lie on a 10 ms grid.

## fig3c-cn-selectivity-abolished

> CN selectivity for lick left and lick right trials was abolished by bilateral ALM
> photo-inhibition.

In [ ]:
def cn_photostim_selectivity(units_, code, rng):
    """As analysis_compiled_all_DCN_ALMsilencing_selectivity.m and analysis_DCN_ALM_photoinhibition_selectivity.m."""
    rows = []
    for u in units_:
        s, i, use = u["s"], u["i"], u["use"]
        t = s["trials"]
        r = cn.epoch_rates(s, i)
        ctrl, stim = use & (t.stim.values == 0), use & (t.stim.values == code)
        R, L = (t.trial_type == "R").values, (t.trial_type == "L").values
        if (ctrl & R).sum() < 30 or (ctrl & L).sum() < 30 or (stim & R).sum() <= 2 or (stim & L).sum() <= 2:
            continue
        ps = stats.ttest_ind(r["sample"].values[ctrl & R], r["sample"].values[ctrl & L]).pvalue
        pd_ = stats.ttest_ind(r["delay"].values[ctrl & R], r["delay"].values[ctrl & L]).pvalue
        if not (ps < 0.05 or pd_ < 0.05):
            continue
        ri, li = rng.permutation(np.flatnonzero(ctrl & R)), rng.permutation(np.flatnonzero(ctrl & L))
        d = r["delay"].values
        sgn = 1.0 if d[ri[:20]].mean() > d[li[:20]].mean() else (-1.0 if d[ri[:20]].mean() < d[li[:20]].mean() else 0.0)
        if sgn == 0:
            continue
        rows.append({"mouse": u["mouse"], "location": u["location"],
                     "ctrl": sgn * (d[ri[20:]].mean() - d[li[20:]].mean()), "stim": sgn * (d[stim & R].mean() - d[stim & L].mean())})
    return pd.DataFrame(rows)


def judge(tbl, rng, label):
    ratio = tbl.stim.mean() / tbl.ctrl.mean()
    boot = np.array([tbl.iloc[rng.integers(0, len(tbl), len(tbl))].pipe(lambda x: x.stim.mean() - x.ctrl.mean()) for _ in range(10000)])
    p = float((boot >= 0).mean())
    print(f"{label}: {len(tbl)} neurons; delay selectivity control {tbl.ctrl.mean():.2f}, photo-inhibition {tbl.stim.mean():.2f} "
          f"spikes/s; ratio {ratio:.2f}; one-tailed bootstrap p (not lower) = {p:.4f}")
    return ratio, p

In [ ]:
rng = np.random.default_rng(604)
sel9 = cn_photostim_selectivity(vgat_units, 9, rng)
judge(sel9, rng, "Bilateral ALM photo-inhibition (code 9)")
print("Expected n = 157")
print("By nucleus:"); print(sel9.groupby("location")[["ctrl", "stim"]].agg(["mean", "size"]).round(2).to_string())
print("By mouse, ratio:", sel9.groupby("mouse").apply(lambda x: round(x.stim.mean() / x.ctrl.mean(), 2), include_groups=False).to_dict())

**Result: reproduced.** In 137 selective CN units (against 157), delay selectivity falls from
3.73 spikes/s on control to 0.68 on bilateral ALM photo-inhibition trials, a ratio of 0.18 (one-
tailed p < 0.0001). It falls to 0.01 in fastigial units and to 0.96 and 1.15 in dentate and
interposed units.

## ed6-cn-either-hemisphere

> Photo-inhibiting either side of ALM reduced CN preparatory activity.

In [ ]:
rng = np.random.default_rng(605)
for code, lab in ((7, "left ALM (ipsilateral to the CN recording, code 7)"), (8, "right ALM (contralateral, code 8)"),
                  (10, "variant, bilateral M1 (code 10)")):
    us = cn_units_with({code})
    tbl = cn_photostim_selectivity(us, code, rng)
    if len(tbl):
        judge(tbl, rng, lab)
    else:
        print(lab, ": no neurons meet the trial criteria")

**Result: reproduced.** Photo-inhibiting the left ALM lowers CN delay selectivity from 3.48 to
1.29 spikes/s (154 units, p < 0.0001) and the right ALM from 3.37 to 1.56 (152 units, p =
0.0002). Bilateral M1 photo-inhibition, a variant, lowers it from 3.62 to 1.84 (75 units, p =
0.019), a smaller reduction than bilateral ALM photo-inhibition (ratio 0.51 against 0.18 in
fig3c-cn-selectivity-abolished).

## ed4c-lick-modulation (blocked)

> In ALM and CN a similar small proportion (below 10%) of neurons modulated their spike rates
> during individual licks.

The dandiset has no lick times. For the ALM sessions they are in the MATLAB record of the paper
on Zenodo, whose data files also hold the raw voltage traces. The time to fetch them at the rate
measured above:

In [ ]:
import requests
rec = requests.get("https://zenodo.org/api/records/6653158", timeout=60).json()
mat = [(f["key"], f["size"]) for f in rec["files"] if f["key"].startswith("datafiles_")]
total = sum(size for _, size in mat)
rate = timing["compressed_bytes"] / timing["seconds"]
print("MATLAB data files of doi:10.5281/zenodo.6653158:", [(k, round(v / 1e9, 2)) for k, v in mat])
print(f"total {total / 1e9:.1f} GB; at the {rate / 1e6:.1f} MB/s measured here, {total / rate / 60:.0f} min to fetch, "
      "against the 30 min limit for a notebook")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2-cn-unit-counts | not reproduced |
| fig2-cn-selective-fraction | not reproduced |
| fig2-cn-preparatory-fraction | partially reproduced |
| fig2-cn-perimovement-fraction | not reproduced |
| ed4e-cn-predict-movement | not reproduced |
| ed4g-fn-contra-late-delay | reproduced |
| fig3b-cn-modulated-fraction | not reproduced |
| fig3b-cn-net-increase | reproduced |
| fig3-cn-latency | not reproduced |
| fig3c-cn-selectivity-abolished | reproduced |
| ed6-cn-either-hemisphere | reproduced |
| ed4c-lick-modulation | blocked, too large |

**How the analysis differs from the paper's.** The Zenodo record holds more sessions (185 against
169) and more units (757 in the CN against 564) than the paper reports, and all of them are used.
With the scripts' rule of at least 30 correct trials of each type, used there for the population
analyses, the selective, preparatory and peri-movement fractions match the paper (0.757, 0.372 and
0.670), so the paper's counts likely rest on a trial-count selection the Methods do not state; the
outcomes are judged on all units, as the criteria fixed. The sessions of the 2016 mice have a 1.7 s
delay epoch.

**What was found.** The paper's direction results hold in these data: bilateral ALM photo-inhibition
removes most CN delay selectivity and increases more CN neurons than it decreases, and either ALM
hemisphere reduces it. The fractions of selective and modulated neurons are lower than the paper's
on the full set of units. One pair of units in one session (BAYLORNL10, 2017-05-03) has identical
spike trains.